# Train and Evaluate Tutorial - Stocks

This notebook covers feature engineering, reward schemes, and training a DQN agent on stock trading data.

## 📚 Related Tutorials

Before diving in, review these essential tutorials:

| Tutorial | Description |
|----------|-------------|
| [First Training](../docs/tutorials/04-training/01-first-training.md) | Getting started with training |
| [Reward Schemes](../docs/tutorials/03-components/02-reward-schemes.md) | Why PBR works and reward design |
| [Common Failures](../docs/tutorials/02-domains/track-b-rl-for-traders/02-common-failures.md) | **Critical pitfalls to avoid** |
| [Overfitting](../docs/tutorials/05-advanced/01-overfitting.md) | Detection and prevention |
| [Commission Analysis](../docs/tutorials/05-advanced/02-commission.md) | Key finding: commission destroys profits |

### ⚠️ Important Warning

Our experiments show that agents **can predict market direction** but **overtrading destroys profits** when commission is applied. See the [Commission Analysis](../docs/tutorials/05-advanced/02-commission.md) tutorial for details.

---

## Define global variables

In [ ]:
n_steps = 1000
n_episodes = 20
window_size = 30
memory_capacity = n_steps * 10
save_path = 'agents/'
n_bins = 5
seed = 1337
learning_rate = 1e-3
gamma = 0.99
eps_decay = 200
reward_c_up = 2.0
reward_beta = 0.2

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU enabled: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('No GPU found, using CPU')

## Setup Data Fetching

In [ ]:
import yfinance as yf

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning, module="pandas")
pd.set_option('future.no_silent_downcasting', True)

def prepare_data(df):
    df['volume'] = np.int64(df['volume'])
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values(by='date', ascending=True, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['date'] = df['date'].dt.strftime('%Y-%m-%d %I:%M %p')
    return df

def fetch_data(ticker='AAPL', period='2y', interval='1h'):
    stock = yf.Ticker(ticker)  # type: ignore[no-any-return]
    df = stock.history(period=period, interval=interval)  # type: ignore[no-any-return]
    df = df.reset_index()
    df = df.rename(columns={
        'Date': 'date',
        'Datetime': 'date',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume'
    })
    df = df[['date', 'open', 'high', 'low', 'close', 'volume']]
    df = prepare_data(df)
    return df

In [ ]:
data = fetch_data('AAPL')
data

## Create features for the feed module

In [ ]:
import ta
import ta.trend  # type: ignore[no-redef]

def fix_dataset_inconsistencies(dataframe, fill_value=None):
    dataframe = dataframe.replace([-np.inf, np.inf], np.nan)

    if fill_value is None:
        dataframe.iloc[0,:] = \
            dataframe.apply(lambda column: column.iloc[column.first_valid_index()], axis='index')
    else:
        dataframe.iloc[0,:] = \
            dataframe.iloc[0,:].fillna(fill_value)

    return dataframe.ffill().dropna(axis='columns')

def rsi(price: 'pd.Series', period: float) -> 'pd.Series':
    r = price.diff()
    upside = r.clip(lower=0)
    downside = -r.clip(upper=0)
    rs = upside.ewm(alpha=1 / period).mean() / downside.ewm(alpha=1 / period).mean()
    return 100*(1 - (1 + rs) ** -1)

def macd(price: 'pd.Series', fast: float, slow: float, signal: float) -> 'pd.Series':
    fm = price.ewm(span=fast, adjust=False).mean()
    sm = price.ewm(span=slow, adjust=False).mean()
    md = fm - sm
    signal = md - md.ewm(span=signal, adjust=False).mean()
    return signal

def generate_features(data):
    df = data.copy()

    ta.add_all_ta_features(df, 
                            'open', 
                            'high', 
                            'low', 
                            'close', 
                            'volume', 
                            fillna=True)

    df = df.rename(columns={'open': 'Open', 
                                'high': 'High', 
                                'low': 'Low', 
                                'close': 'Close', 
                                'volume': 'Volume'})
    df = df.set_index('date')

    features = pd.DataFrame.from_dict({
        'prev_open': df['Open'].shift(1),
        'prev_high': df['High'].shift(1),
        'prev_low': df['Low'].shift(1),
        'prev_close': df['Close'].shift(1),
        'prev_volume': df['Volume'].shift(1),
        'vol_5': df['Close'].rolling(window=5).std().abs(),
        'vol_10': df['Close'].rolling(window=10).std().abs(),
        'vol_20': df['Close'].rolling(window=20).std().abs(),
        'vol_30': df['Close'].rolling(window=30).std().abs(),
        'vol_50': df['Close'].rolling(window=50).std().abs(),
        'vol_60': df['Close'].rolling(window=60).std().abs(),
        'vol_100': df['Close'].rolling(window=100).std().abs(),
        'vol_200': df['Close'].rolling(window=200).std().abs(),
        'ma_5': df['Close'].rolling(window=5).mean(),
        'ma_10': df['Close'].rolling(window=10).mean(),
        'ma_20': df['Close'].rolling(window=20).mean(),
        'ma_30': df['Close'].rolling(window=30).mean(),
        'ma_50': df['Close'].rolling(window=50).mean(),
        'ma_60': df['Close'].rolling(window=60).mean(),
        'ma_100': df['Close'].rolling(window=100).mean(),
        'ma_200': df['Close'].rolling(window=200).mean(),
        'ema_5': ta.trend.ema_indicator(df['Close'], window=5, fillna=True),
        'ema_10': ta.trend.ema_indicator(df['Close'], window=10, fillna=True),
        'ema_20': ta.trend.ema_indicator(df['Close'], window=20, fillna=True),
        'ema_60': ta.trend.ema_indicator(df['Close'], window=60, fillna=True),
        'ema_64': ta.trend.ema_indicator(df['Close'], window=64, fillna=True),
        'ema_120': ta.trend.ema_indicator(df['Close'], window=120, fillna=True),
        'lr_open': np.log(df['Open']).diff().fillna(0),
        'lr_high': np.log(df['High']).diff().fillna(0),
        'lr_low': np.log(df['Low']).diff().fillna(0),
        'lr_close': np.log(df['Close']).diff().fillna(0),
        'r_volume': df['Close'].diff().fillna(0),
        'rsi_5': rsi(df['Close'], period=5),
        'rsi_10': rsi(df['Close'], period=10),
        'rsi_100': rsi(df['Close'], period=100),
        'rsi_7': rsi(df['Close'], period=7),
        'rsi_28': rsi(df['Close'], period=28),
        'rsi_6': rsi(df['Close'], period=6),
        'rsi_14': rsi(df['Close'], period=14),
        'rsi_26': rsi(df['Close'], period=24),
        'macd_normal': macd(df['Close'], fast=12, slow=26, signal=9),
        'macd_short': macd(df['Close'], fast=10, slow=50, signal=5),
        'macd_long': macd(df['Close'], fast=200, slow=100, signal=50),
    })

    data = pd.concat([df, features], axis='columns').ffill()

    data = data.loc[:,~data.columns.duplicated()]

    data = data.rename(columns={'Open': 'open', 
                                'High': 'high', 
                                'Low': 'low', 
                                'Close': 'close', 
                                'Volume': 'volume'})

    data = data.reset_index()

    data = data.iloc[200:]
    data = data.reset_index(drop=True)

    data = fix_dataset_inconsistencies(data, fill_value=None)
    return data

In [ ]:
data = generate_features(data)
data

## Remove features with low variance before splitting the dataset

In [ ]:
from sklearn.feature_selection import VarianceThreshold
sel = VarianceThreshold(threshold=(.8 * (1 - .8)))
date = data[['date']].copy()
data = data.drop(columns=['date'])
sel.fit(data)
data = data[data.columns[sel.get_support(indices=True)]]
data = pd.concat([date, data], axis='columns')
data

## Split dataset

In [ ]:
from sklearn.model_selection import train_test_split

def split_data(data):
    X_train_test, X_valid, y_train_test, y_valid = \
        train_test_split(data, data['close'].pct_change(), train_size=0.67, test_size=0.33, shuffle=False)

    X_train, X_test, y_train, y_test = \
        train_test_split(X_train_test, y_train_test, train_size=0.50, test_size=0.50, shuffle=False)

    return X_train, X_test, X_valid, y_train, y_test, y_valid

In [ ]:
import os
X_train, X_test, X_valid, y_train, y_test, y_valid = \
    split_data(data)

cwd = os.getcwd()
train_csv = os.path.join(cwd, 'train.csv')
test_csv = os.path.join(cwd, 'test.csv')
valid_csv = os.path.join(cwd, 'valid.csv')
X_train.to_csv(train_csv, index=False)
X_test.to_csv(test_csv, index=False)
X_valid.to_csv(valid_csv, index=False)

## Get dataset statistics

In [ ]:
from scipy.stats import iqr

def estimate_outliers(data):
    return iqr(data) * 1.5

def estimate_percent_gains(data, column='close'):
    returns = get_returns(data, column=column)
    gains = estimate_outliers(returns)
    return gains

def get_returns(data, column='close'):
    return fix_dataset_inconsistencies(data[[column]].pct_change(), fill_value=0)

def precalculate_ground_truths(data, column='close', threshold=None):
    returns = get_returns(data, column=column)
    gains = estimate_outliers(returns) if threshold is None else threshold
    binary_gains = (returns[column] > gains).astype(int)
    return binary_gains

def is_null(data):
    return data.isnull().sum().sum() > 0

def is_sparse(data, column='close'):
    binary_gains = precalculate_ground_truths(data, column=column)
    bins = [n * (binary_gains.shape[0] // n_bins) for n in range(n_bins)]
    bins += [binary_gains.shape[0]]
    bins = [binary_gains.iloc[bins[n]:bins[n + 1]] for n in range(n_bins)]
    return all([bin.astype(bool).any() for bin in bins])

def is_data_predictible(data, column):
    return not is_null(data) and is_sparse(data, column)

data.describe(include='all')

## Evaluate outlier sparsity of the data

In [ ]:
import matplotlib.pyplot as plt
plt.plot(get_returns(data, column='close'))
plt.show()
is_data_predictible(data, 'close')

## Percentage of the dataset generating rewards (keep between 5% to 15% or just rely on is_data_predictible())

In [ ]:
plt.plot(precalculate_ground_truths(data, column='close').iloc[:1000])
plt.show()
percent_rewardable = round(precalculate_ground_truths(data, column='close').mean() * 100, 2)
print(percent_rewardable)

## Threshold to pass to AnomalousProfit reward scheme

In [ ]:
X_train_test = pd.concat([X_train, X_test], axis='index')
threshold = estimate_percent_gains(X_train, 'close')
threshold

## Implement basic feature engineering

In [ ]:
from sklearn.ensemble import RandomForestClassifier

from feature_engine.selection import SelectBySingleFeaturePerformance

In [ ]:
rf = RandomForestClassifier(n_estimators=10, 
                            random_state=seed, 
                            n_jobs=6)

sel = SelectBySingleFeaturePerformance(variables=None, 
                                       estimator=rf, 
                                       scoring="roc_auc", 
                                       cv=5, 
                                       threshold=0.5)

sel.fit(X_train, precalculate_ground_truths(X_train, column='close'))

In [ ]:
feature_performance = pd.Series(sel.feature_performance_).sort_values(ascending=False)
feature_performance

In [ ]:
feature_performance.plot.bar(figsize=(20, 5))
plt.title('Performance of ML models trained with individual features')
plt.ylabel('roc-auc')

In [ ]:
features_to_drop = sel.features_to_drop_
features_to_drop

In [ ]:
to_drop = list(set(features_to_drop) - set(['open', 'high', 'low', 'close', 'volume']))
len(to_drop)

In [ ]:
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)
X_valid = X_valid.drop(columns=to_drop)

X_train.shape, X_test.shape, X_valid.shape

In [ ]:
X_train.columns.tolist()

## Normalize the dataset subsets to make the model converge faster

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_type = MinMaxScaler

def get_feature_scalers(X, scaler_type=scaler_type):
    scalers = []
    for name in list(X.columns[X.columns != 'date']):
        scalers.append(scaler_type().fit(X[name].values.reshape(-1, 1)))
    return scalers

def get_scaler_transforms(X, scalers):
    X_scaled = []
    for name, scaler in zip(list(X.columns[X.columns != 'date']), scalers):
        X_scaled.append(scaler.transform(X[name].values.reshape(-1, 1)))
    X_scaled = pd.concat([pd.DataFrame(column, columns=[name]) for name, column in \
                          zip(list(X.columns[X.columns != 'date']), X_scaled)], axis='columns')
    return X_scaled

def normalize_data(X_train, X_test, X_valid):
    X_train_test = pd.concat([X_train, X_test], axis='index')
    X_train_test_valid = pd.concat([X_train_test, X_valid], axis='index')

    X_train_test_dates = X_train_test[['date']]
    X_train_test_valid_dates = X_train_test_valid[['date']]

    X_train_test = X_train_test.drop(columns=['date'])
    X_train_test_valid = X_train_test_valid.drop(columns=['date'])

    train_test_scalers = \
        get_feature_scalers(X_train_test, 
                            scaler_type=scaler_type)

    X_train_test_scaled = \
        get_scaler_transforms(X_train_test, 
                              train_test_scalers)
    X_train_test_valid_scaled = \
        get_scaler_transforms(X_train_test_valid, 
                              train_test_scalers)

    X_train_test_scaled = \
        pd.concat([X_train_test_dates, 
                   X_train_test_scaled], 
                  axis='columns')
    X_train_test_valid_scaled = \
        pd.concat([X_train_test_valid_dates, 
                   X_train_test_valid_scaled], 
                  axis='columns')

    X_train_scaled = X_train_test_scaled.iloc[:X_train.shape[0]]
    X_test_scaled = X_train_test_scaled.iloc[X_train.shape[0]:]
    X_valid_scaled = X_train_test_valid_scaled.iloc[X_train_test.shape[0]:]

    return (train_test_scalers, 
            X_train_scaled, 
            X_test_scaled, 
            X_valid_scaled)

In [ ]:
train_test_scalers, X_train_scaled, X_test_scaled, X_valid_scaled = \
    normalize_data(X_train, X_test, X_valid)

## Write a reward scheme encouraging rare volatile upside trades

In [ ]:
from tensortrade.env.default.rewards import TensorTradeRewardScheme


class AnomalousProfit(TensorTradeRewardScheme):
    """A simple reward scheme that rewards the agent for exceeding a 
    precalculated percentage in the net worth.

    Parameters
    ----------
    threshold : float
        The minimum value to exceed in order to get the reward.

    Attributes
    ----------
    threshold : float
        The minimum value to exceed in order to get the reward.
    """

    registered_name = "anomalous"

    def __init__(self, threshold: float = 0.02, window_size: int = 1,
                 c_up: float = 2.0, beta: float = 0.2, c_down: float = 1.0):
        self._window_size = self.default('window_size', window_size)
        self._threshold = self.default('threshold', threshold)
        self._c_up = self.default('c_up', c_up)
        self._beta = self.default('beta', beta)
        self._c_down = self.default('c_down', c_down)

    def get_reward(self, portfolio: 'Portfolio') -> float:  # type: ignore[arg-type]
        """Rewards the agent for anomalous upside moves in net worth.

        Only net-worth returns that exceed the anomaly threshold ``tau`` are
        rewarded (scaled, capped at ``c_up``). Small/moderate upside and flat
        steps are neutral, while downside incurs a light, capped penalty.

        Parameters
        ----------
        portfolio : `Portfolio`
            The portfolio being used by the environment.

        Returns
        -------
        float
            The shaped reward for the last step.
        """
        performance = pd.DataFrame.from_dict(portfolio.performance).T
        if performance.shape[0] < 2:
            return 0.0

        net_worths = performance['net_worth']
        last_return = net_worths.iloc[-1] / net_worths.iloc[-2] - 1

        tau = self._threshold

        if last_return > tau:
            reward = min(last_return / tau, self._c_up)
        elif last_return >= 0:
            reward = 0.0
        else:
            reward = max(-self._beta * abs(last_return) / tau, -self._c_down)

        return float(reward)

In [ ]:
class PenalizedProfit(TensorTradeRewardScheme):
    """A reward scheme which penalizes net worth loss and 
    decays with the time spent.

    Parameters
    ----------
    cash_penalty_proportion : float
        cash_penalty_proportion

    Attributes
    ----------
    cash_penalty_proportion : float
        cash_penalty_proportion.
    """

    registered_name = "penalized"

    def __init__(self, cash_penalty_proportion: float = 0.10):
        self._cash_penalty_proportion = \
            self.default('cash_penalty_proportion', 
                         cash_penalty_proportion)

    def get_reward(self, portfolio: 'Portfolio') -> float:  # type: ignore[arg-type]
        """Rewards the agent for gaining net worth while holding the asset.

        Parameters
        ----------
        portfolio : `Portfolio`
            The portfolio being used by the environment.

        Returns
        -------
        int
            A penalized reward.
        """
        performance = pd.DataFrame.from_dict(portfolio.performance).T
        current_step = performance.shape[0]
        if current_step > 1:
            initial_amount = portfolio.initial_net_worth
            net_worth = performance['net_worth'].iloc[-1]
            cash_worth = performance['nasdaq:/USD:/total'].iloc[-1]
            cash_penalty = max(0, (net_worth * self._cash_penalty_proportion - cash_worth))
            net_worth -= cash_penalty
            reward = (net_worth / initial_amount) - 1
            reward /= current_step
            return reward
        else:
            return 0.0

## Setup Trading Environment

In [ ]:
import tensortrade.env.default as default

from tensortrade.feed.core import DataFeed, Stream
from tensortrade.feed.core.base import NameSpace
from tensortrade.env.default.actions import BSH
from tensortrade.oms.exchanges import Exchange, ExchangeOptions
from tensortrade.oms.services.execution.simulated import execute_order
from tensortrade.oms.instruments import USD, AAPL
from tensortrade.oms.wallets import Wallet, Portfolio

commission = 0.001
price = Stream.source(list(X_train["close"]), 
                      dtype="float").rename("USD-AAPL")
nasdaq = Exchange("nasdaq", 
                  service=execute_order, options=ExchangeOptions(commission=commission))(price)

cash = Wallet(nasdaq, 50000 * USD)
asset = Wallet(nasdaq, 0 * AAPL)

portfolio = Portfolio(USD, [cash, asset])

with NameSpace("nasdaq"):
    features = [
        Stream.source(list(X_train_scaled[c]), 
                      dtype="float").rename(c) for c in X_train_scaled.columns[1:]
    ]

feed: DataFeed = DataFeed(features)
feed.compile()

renderer_feed = DataFeed([
    Stream.source(list(X_train["date"])).rename("date"),
    Stream.source(list(X_train["open"]), dtype="float").rename("open"),
    Stream.source(list(X_train["high"]), dtype="float").rename("high"),
    Stream.source(list(X_train["low"]), dtype="float").rename("low"),
    Stream.source(list(X_train["close"]), dtype="float").rename("close"), 
    Stream.source(list(X_train["volume"]), dtype="float").rename("volume") 
])

action_scheme = BSH(
    cash=cash,
    asset=asset
)

reward_scheme = AnomalousProfit(threshold=threshold, c_up=reward_c_up, beta=reward_beta)

env = default.create(
    portfolio=portfolio,
    action_scheme=action_scheme,
    reward_scheme=reward_scheme,
    feed=feed,
    renderer_feed=renderer_feed,
    renderer=default.renderers.PlotlyTradingChart(),
    window_size=30
)

In [ ]:
env.observer.feed.next()

## Setup and Train DQN Agent

In [ ]:
def get_optimal_batch_size(window_size=30, n_steps=1000, batch_factor=4, stride=1):
    """
    lookback = 30
    batch_factor = 4
    stride = 1
    """
    lookback = window_size
    sample_size = n_steps
    batch_size = ((sample_size - lookback - stride) // batch_factor)
    return batch_size

batch_size = get_optimal_batch_size(window_size=window_size, n_steps=n_steps, batch_factor=4)
batch_size

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import namedtuple, deque
import random

DQNTransition = namedtuple('DQNTransition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    
    def push(self, *args):
        self.memory.append(DQNTransition(*args))
    
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    
    def __len__(self):
        return len(self.memory)

class DQNNetwork(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(DQNNetwork, self).__init__()
        n_features = input_shape[1] if len(input_shape) > 1 else input_shape[0]
        self.conv1 = nn.Conv1d(n_features, 32, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=4, stride=2, padding=1)
        
        self.prelu1 = nn.PReLU()
        self.prelu2 = nn.PReLU()
        self.prelu3 = nn.PReLU()
        
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(64)
        
        self.gru = nn.GRU(64, 64, batch_first=True)
        
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, n_actions)
        
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu1(x)
        x = self.dropout(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.prelu2(x)
        x = self.dropout(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.prelu3(x)
        x = self.dropout(x)
        
        x = x.permute(0, 2, 1)
        _, h_n = self.gru(x)
        x = h_n.squeeze(0)
        
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

class PyTorchDQNAgent:
    def __init__(self, env, device='cuda'):
        self.env = env
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.n_actions = int(env.action_space.n)
        self.observation_shape = env.observation_space.shape
        
        self.policy_net = DQNNetwork(self.observation_shape, self.n_actions).to(self.device)
        self.target_net = DQNNetwork(self.observation_shape, self.n_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.001)
        
    def get_action(self, state, epsilon=0.0):
        if random.random() < epsilon:
            return random.randint(0, self.n_actions - 1)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_t)
                return q_values.argmax(dim=1).item()
    
    def optimize_model(self, batch_size, gamma=0.99):
        if len(self.memory) < batch_size:
            return
        
        transitions = self.memory.sample(batch_size)
        batch = DQNTransition(*zip(*transitions))
        
        state_batch = torch.FloatTensor(np.array(batch.state)).to(self.device)
        action_batch = torch.LongTensor(batch.action).to(self.device)
        reward_batch = torch.FloatTensor(batch.reward).to(self.device)
        next_state_batch = torch.FloatTensor(np.array(batch.next_state)).to(self.device)
        done_batch = torch.FloatTensor(np.array(batch.done, dtype=np.float32)).to(self.device)
        
        state_action_values = self.policy_net(state_batch).gather(1, action_batch.unsqueeze(1))
        
        with torch.no_grad():
            next_state_values = self.target_net(next_state_batch).max(1)[0]
            expected_state_action_values = (next_state_values * gamma * (1 - done_batch)) + reward_batch
        
        loss = nn.SmoothL1Loss()(state_action_values.squeeze(), expected_state_action_values)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    def train(self, n_steps=1000, n_episodes=20, batch_size=32, memory_capacity=10000, 
              save_path='agents/', gamma=0.99, eps_start=0.9, eps_end=0.05, eps_decay=200):
        self.memory = ReplayMemory(memory_capacity)
        steps_done = 0
        
        for episode in range(n_episodes):
            state = self.env.reset()
            if isinstance(state, tuple):
                state = state[0]
            
            episode_reward = 0
            episode_losses = []
            
            for step in range(n_steps):
                eps_threshold = eps_end + (eps_start - eps_end) * \
                    np.exp(-steps_done / eps_decay)
                
                action = self.get_action(state, eps_threshold)
                
                result = self.env.step(action)
                if len(result) == 5:
                    next_state, reward, terminated, truncated, _ = result
                    done = terminated or truncated
                else:
                    next_state, reward, done, _ = result
                
                self.memory.push(state, action, reward, next_state, done)
                
                state = next_state
                episode_reward += reward
                steps_done += 1
                
                loss = self.optimize_model(batch_size, gamma)
                if loss is not None:
                    episode_losses.append(loss)
                
                if steps_done % 100 == 0:
                    self.target_net.load_state_dict(self.policy_net.state_dict())
                
                if done:
                    break
            
            avg_loss = np.mean(episode_losses) if episode_losses else 0
            print(f'Episode {episode + 1}/{n_episodes}, Reward: {episode_reward:.2f}, Loss: {avg_loss:.4f}, Steps: {step + 1}')
            
            if save_path and (episode + 1) % 5 == 0:
                torch.save(self.policy_net.state_dict(), f'{save_path}dqn_model_ep{episode + 1}.pth')

agent = PyTorchDQNAgent(env, device='cuda')
agent.optimizer = torch.optim.Adam(agent.policy_net.parameters(), lr=learning_rate)

os.makedirs(save_path, exist_ok=True)
agent.train(n_steps=n_steps, 
            n_episodes=n_episodes, 
            batch_size=batch_size, 
            memory_capacity=memory_capacity, 
            save_path=save_path,
            gamma=gamma,
            eps_decay=eps_decay)

## Implement validation here

In [ ]:
def validate_agent(agent, X_data, X_data_scaled, window_size=30):
    from tensortrade.feed.core import DataFeed, Stream
    from tensortrade.feed.core.base import NameSpace
    from tensortrade.oms.exchanges import ExchangeOptions
    
    price_stream = Stream.source(list(X_data["close"]), dtype="float").rename("USD-AAPL")
    val_exchange = Exchange("nasdaq", service=execute_order, options=ExchangeOptions(commission=commission))(price_stream)
    val_cash = Wallet(val_exchange, 50000 * USD)
    val_asset = Wallet(val_exchange, 0 * AAPL)
    val_portfolio = Portfolio(USD, [val_cash, val_asset])
    
    with NameSpace("nasdaq"):
        val_features = [
            Stream.source(list(X_data_scaled[c]), dtype="float").rename(c)
            for c in X_data_scaled.drop(columns=['date'], errors='ignore').columns
        ]
    
    val_feed: DataFeed = DataFeed(val_features)
    val_feed.compile()
    
    val_renderer_feed = DataFeed([
        Stream.source(list(X_data["date"])).rename("date"),
        Stream.source(list(X_data["open"]), dtype="float").rename("open"),
        Stream.source(list(X_data["high"]), dtype="float").rename("high"),
        Stream.source(list(X_data["low"]), dtype="float").rename("low"),
        Stream.source(list(X_data["close"]), dtype="float").rename("close"),
        Stream.source(list(X_data["volume"]), dtype="float").rename("volume")
    ])
    
    val_env = default.create(
        portfolio=val_portfolio,
        action_scheme=BSH(cash=val_cash, asset=val_asset),
        reward_scheme=AnomalousProfit(threshold=threshold),
        feed=val_feed,
        renderer_feed=val_renderer_feed,
        window_size=window_size
    )
    
    val_agent = PyTorchDQNAgent(val_env, device='cuda')
    val_agent.policy_net.load_state_dict(agent.policy_net.state_dict())
    val_agent.policy_net.eval()
    val_agent.target_net.load_state_dict(agent.target_net.state_dict())
    val_agent.target_net.eval()
    
    state = val_env.reset()
    if isinstance(state, tuple):
        state = state[0]
    
    net_worth = [val_env.action_scheme.portfolio.initial_net_worth]
    actions = []
    prices = [X_data.iloc[0]['close']]
    
    for i in range(1, len(X_data)):
        action = val_agent.get_action(state, epsilon=0.0)
        step_result = val_env.step(action)
        
        next_state, reward, terminated, truncated, _ = step_result
        done = terminated or truncated
        
        state = next_state
        nw = val_env.action_scheme.portfolio.net_worth
        net_worth.append(nw)
        actions.append(action)
        prices.append(X_data.iloc[i]['close'])
        
        if done:
            break
    
    total_return = (net_worth[-1] / net_worth[0] - 1) * 100
    buy_hold_return = (X_data.iloc[-1]['close'] / X_data.iloc[0]['close'] - 1) * 100
    
    nw_series = pd.Series(net_worth)
    returns = nw_series.pct_change().dropna()
    sharpe = np.sqrt(252 * 6.5) * returns.mean() / returns.std() if returns.std() > 0 else 0
    
    trades = sum(1 for i in range(1, len(actions)) if actions[i] != actions[i-1])
    
    print(f'Total Return: {total_return:.2f}%')
    print(f'Buy & Hold Return: {buy_hold_return:.2f}%')
    print(f'Sharpe Ratio: {sharpe:.2f}')
    print(f'Total Trades: {trades}')
    print(f'Net Worth: {net_worth[0]:.2f} -> {net_worth[-1]:.2f}')
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    axes[0].plot(net_worth, label='Portfolio Value', color='blue')
    axes[0].axhline(y=net_worth[0], color='gray', linestyle='--', alpha=0.5)
    axes[0].set_title('Portfolio Net Worth')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(prices, label='Close Price', color='green')
    axes[1].set_title('Asset Price')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return net_worth, actions, total_return

net_worth, trade_actions, test_return = validate_agent(agent, X_test, X_test_scaled, window_size=window_size)

net_worth_val, trade_actions_val, valid_return = validate_agent(agent, X_valid, X_valid_scaled, window_size=window_size)

In [ ]:
# Trains one agent per hyperparameter combination on a FRESH environment and
# selects the best by validation net-worth return. The search is intentionally
# small; expand `grid` (e.g. add `beta` / `eps_decay` to itertools.product) to
# cover more. For a quick pass, lower `sweep_episodes` / `n_steps` below.
import itertools
import pandas as pd
from tensortrade.oms.exchanges import ExchangeOptions

sweep_episodes = 10  # fewer than the full run; bump up for a more thorough search


def make_env(reward_kwargs=None, window_size=window_size):
    """Build a fresh default trading env for one trial.

    A new exchange, wallets, portfolio, feed and action/reward schemes are
    constructed on every call so trials do not leak state into each other.
    """
    reward_kwargs = reward_kwargs or {}
    price = Stream.source(list(X_train["close"]), dtype="float").rename("USD-AAPL")
    nasdaq = Exchange("nasdaq", service=execute_order, options=ExchangeOptions(commission=commission))(price)

    cash = Wallet(nasdaq, 50000 * USD)
    asset = Wallet(nasdaq, 0 * AAPL)
    portfolio = Portfolio(USD, [cash, asset])

    with NameSpace("nasdaq"):
        features = [
            Stream.source(list(X_train_scaled[c]), dtype="float").rename(c)
            for c in X_train_scaled.columns[1:]
        ]

    feed = DataFeed(features)
    feed.compile()

    renderer_feed = DataFeed([
        Stream.source(list(X_train["date"])).rename("date"),
        Stream.source(list(X_train["open"]), dtype="float").rename("open"),
        Stream.source(list(X_train["high"]), dtype="float").rename("high"),
        Stream.source(list(X_train["low"]), dtype="float").rename("low"),
        Stream.source(list(X_train["close"]), dtype="float").rename("close"),
        Stream.source(list(X_train["volume"]), dtype="float").rename("volume"),
    ])

    action_scheme = BSH(cash=cash, asset=asset)
    reward_scheme = AnomalousProfit(threshold=threshold, **reward_kwargs)

    env = default.create(
        portfolio=portfolio,
        action_scheme=action_scheme,
        reward_scheme=reward_scheme,
        feed=feed,
        renderer_feed=renderer_feed,
        renderer=default.renderers.EmptyRenderer(),
        window_size=window_size,
    )
    return env


def run_trial(lr, gamma, eps_decay, c_up, beta,
              n_steps=n_steps, n_episodes=sweep_episodes, window_size=window_size):
    """Train one agent and return its validation/test net-worth returns."""
    env = make_env(reward_kwargs={"c_up": c_up, "beta": beta}, window_size=window_size)
    agent = PyTorchDQNAgent(env, device="cuda")
    agent.optimizer = torch.optim.Adam(agent.policy_net.parameters(), lr=lr)
    agent.train(
        n_steps=n_steps,
        n_episodes=n_episodes,
        batch_size=batch_size,
        memory_capacity=memory_capacity,
        save_path=None,
        gamma=gamma,
        eps_decay=eps_decay,
    )
    _, _, valid_return = validate_agent(agent, X_valid, X_valid_scaled, window_size=window_size)
    _, _, test_return = validate_agent(agent, X_test, X_test_scaled, window_size=window_size)
    return {
        "lr": lr, "gamma": gamma, "eps_decay": eps_decay,
        "c_up": c_up, "beta": beta,
        "valid_return": valid_return, "test_return": test_return,
    }


# Small explicit grid: learning_rate x discount x reward c_up.
# Add `beta` / `eps_decay` to itertools.product to tune them too.
grid = list(itertools.product([1e-3, 5e-4], [0.99, 0.95], [1.5, 2.5]))

results = []
for lr, gamma, c_up in grid:
    print(f"Trial lr={lr} gamma={gamma} c_up={c_up} ...")
    results.append(run_trial(lr=lr, gamma=gamma, eps_decay=200, c_up=c_up, beta=0.2))

results_df = (
    pd.DataFrame(results)
    .sort_values("valid_return", ascending=False)
    .reset_index(drop=True)
)
print("\nSweep results (sorted by validation return):")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(
    f"\nBest config: lr={best.lr} gamma={best.gamma} eps_decay=200 "
    f"c_up={best.c_up} beta=0.2 -> valid={best.valid_return:.2f}% "
    f"test={best.test_return:.2f}%"
)

# Retrain the best config (saving checkpoints) and report once more.
best_env = make_env(reward_kwargs={"c_up": float(best.c_up), "beta": 0.2}, window_size=window_size)
best_agent = PyTorchDQNAgent(best_env, device="cuda")
best_agent.optimizer = torch.optim.Adam(best_agent.policy_net.parameters(), lr=float(best.lr))
os.makedirs(save_path, exist_ok=True)
best_agent.train(
    n_steps=n_steps, n_episodes=n_episodes, batch_size=batch_size,
    memory_capacity=memory_capacity, save_path=save_path,
    gamma=float(best.gamma), eps_decay=200,
)
net_worth, trade_actions, test_return = validate_agent(best_agent, X_test, X_test_scaled, window_size=window_size)
net_worth_val, trade_actions_val, valid_return = validate_agent(best_agent, X_valid, X_valid_scaled, window_size=window_size)
print(f"Best agent retrained -> valid={valid_return:.2f}% test={test_return:.2f}%")

# Persist the winning hyperparameters so re-running the training cell uses them.
learning_rate = float(best.lr)
gamma = float(best.gamma)
reward_c_up = float(best.c_up)
reward_beta = 0.2
print(f"Updated globals -> learning_rate={learning_rate} gamma={gamma} reward_c_up={reward_c_up}")


## Print basic quantstats report

In [ ]:
returns = pd.Series(net_worth).pct_change().dropna()

print('===== Quantstats Performance Report =====')
print(f'Start Date: {X_test.iloc[window_size]["date"]}')
print(f'End Date: {X_test.iloc[-1]["date"]}')
print(f'Duration: {len(returns)} periods')
print()
print(f'Total Return: {test_return:.2f}%')
print(f'Buy & Hold Return: {(X_test.iloc[-1]["close"] / X_test.iloc[window_size]["close"] - 1) * 100:.2f}%')
print()
print(f'Cumulative Return: {(pd.Series(net_worth).iloc[-1] / pd.Series(net_worth).iloc[0] - 1) * 100:.2f}%')
print(f'Annual Return: {returns.mean() * 252 * 6.5 * 100:.2f}%')
print(f'Annual Volatility: {returns.std() * np.sqrt(252 * 6.5) * 100:.2f}%')
print(f'Sharpe Ratio: {np.sqrt(252 * 6.5) * returns.mean() / returns.std():.2f}' if returns.std() > 0 else 'Sharpe Ratio: N/A')
print(f'Max Drawdown: {(pd.Series(net_worth) / pd.Series(net_worth).cummax() - 1).min() * 100:.2f}%')
print()
print(f'Best Day: {returns.max() * 100:.2f}%')
print(f'Worst Day: {returns.min() * 100:.2f}%')
print(f'Total Trades: {sum(1 for i in range(1, len(trade_actions)) if trade_actions[i] != trade_actions[i-1])}')